# Chapter 4 — Seller Analysis
Queries `mart_seller_analysis` from BigQuery and exports Plotly chart JSON for the webpage.

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from google.cloud import bigquery
from google.oauth2 import service_account

pio.json.config.default_engine = 'json'  # prevent binary encoding in exports

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
load_dotenv(os.path.join(project_root, '.env'))

project_id  = os.getenv('GCP_PROJECT_ID')
creds_path  = os.path.join(project_root, os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
credentials = service_account.Credentials.from_service_account_file(creds_path)
client      = bigquery.Client(credentials=credentials, project=project_id)

OUT = os.path.join(project_root, 'outputs')
os.makedirs(OUT, exist_ok=True)
print('Connected to BigQuery ✓')

In [2]:
df = client.query(f"""
    SELECT * FROM `{project_id}.olist_raw.mart_seller_analysis`
    ORDER BY total_revenue DESC
""").to_dataframe()
df.head()

,seller_id,seller_state,seller_city,total_orders,total_items_sold,total_revenue,avg_item_price,avg_review_score,positive_reviews,negative_reviews,on_time_deliveries,late_deliveries,on_time_pct,avg_delivery_days,unique_products,unique_customers
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,Guariba,1131,1155,229237.63,198.47,4.13,891,158,1015,133,87.88,14.9,95,1131
1,53243585a1d6dc2643021fd1853d8905,BA,Lauro De Freitas,358,410,222776.05,543.36,4.08,321,51,384,16,93.66,13.3,23,358
2,4a3ca9315b744ce9f8e9374361493884,SP,Ibitinga,1804,2007,202852.32,101.07,3.80,1333,391,1755,216,87.44,14.4,399,1804
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,Sumare,584,585,192842.13,329.64,4.34,497,60,520,59,88.89,13.3,289,584
4,7c67e1448b00f6e969d365cea6b010ab,SP,Itaquaquecetuba,982,1375,189417.67,137.76,3.35,764,402,1235,131,89.82,22.3,198,982


In [3]:
# Chart 1 — Top 20 Sellers by Revenue
top_sellers = df.head(20).copy()
top_sellers['seller_label'] = top_sellers['seller_id'].str[:8] + '...'

fig1 = px.bar(
    top_sellers, x='total_revenue', y='seller_label', orientation='h',
    title='Top 20 Sellers by Revenue (BRL)',
    labels={'total_revenue': 'Revenue (BRL)', 'seller_label': 'Seller'},
    color='avg_review_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5]
)
fig1.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'})
fig1.show()
with open(os.path.join(OUT, 'seller_top20_revenue.json'), 'w') as f:
    f.write(fig1.to_json())
print('Exported seller_top20_revenue.json')

Exported seller_top20_revenue.json


In [4]:
# Chart 2 — Seller Count and Revenue by State
by_state = df.groupby('seller_state').agg(
    seller_count=('seller_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    avg_review=('avg_review_score', 'mean'),
    avg_on_time=('on_time_pct', 'mean')
).reset_index().sort_values('total_revenue', ascending=False)

fig2 = px.bar(
    by_state, x='seller_state', y='total_revenue',
    title='Total Seller Revenue by State (BRL)',
    labels={'seller_state': 'State', 'total_revenue': 'Revenue (BRL)'},
    color='seller_count',
    color_continuous_scale='Blues'
)
fig2.update_layout(template='plotly_white')
fig2.show()
with open(os.path.join(OUT, 'seller_revenue_by_state.json'), 'w') as f:
    f.write(fig2.to_json())
print('Exported seller_revenue_by_state.json')

Exported seller_revenue_by_state.json


In [5]:
# Chart 3 — Review Score vs On-Time % (top 200 sellers by revenue)
top200 = df.head(200)
fig3 = px.scatter(
    top200, x='on_time_pct', y='avg_review_score',
    size='total_revenue',
    color='seller_state',
    title='Review Score vs On-Time % (Top 200 Sellers)',
    labels={'on_time_pct': 'On-Time %', 'avg_review_score': 'Avg Review Score', 'total_revenue': 'Revenue'}
)
fig3.update_layout(template='plotly_white')
fig3.show()
with open(os.path.join(OUT, 'seller_score_vs_ontime.json'), 'w') as f:
    f.write(fig3.to_json())
print('Exported seller_score_vs_ontime.json')

Exported seller_score_vs_ontime.json


In [6]:
# Chart 4 — Seller State Distribution (pie)
fig4 = px.pie(
    by_state, names='seller_state', values='seller_count',
    title='Seller Distribution by State',
    hole=0.4
)
fig4.update_layout(template='plotly_white')
fig4.show()
with open(os.path.join(OUT, 'seller_state_distribution.json'), 'w') as f:
    f.write(fig4.to_json())
print('Exported seller_state_distribution.json')

Exported seller_state_distribution.json
